# Homework #2 – F1 Data Analysis with PySpark

This notebook answers all six questions using PySpark on the F1 datasets.  
For each question, I include:
1. a markdown cell explaining my logic,
2. the PySpark code used to answer the question,
3. a markdown cell explaining how the code works,
4. and an extra-credit note describing an alternative way to solve the problem.

## 0. Setup

I first load the datasets needed for this assignment and convert key columns into the correct data types.

Because CSV files are often read as strings by default, I explicitly cast date and numeric columns before doing calculations. This helps avoid errors in aggregation, ranking, and joins later in the notebook.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# ---------- LOAD DATA ----------
# ---------- CLEAN / CAST TYPES ----------
df_drivers = (
    spark.read.csv("/Volumes/gr5069/raw/f1_data/drivers.csv", header=True)
    .withColumn("driverId", F.col("driverId").cast("int"))
    .withColumn("dob", F.to_date("dob", "yyyy-MM-dd"))
)

df_races = (
    spark.read.csv("/Volumes/gr5069/raw/f1_data/races.csv", header=True)
    .withColumn("raceId", F.col("raceId").cast("int"))
    .withColumn("year", F.col("year").cast("int"))
    .withColumn("round", F.col("round").cast("int"))
    .withColumn("date", F.to_date("date", "yyyy-MM-dd"))
)

df_results = (
    spark.read.csv("/Volumes/gr5069/raw/f1_data/results.csv", header=True)
    .withColumn("resultId", F.col("resultId").cast("int"))
    .withColumn("raceId", F.col("raceId").cast("int"))
    .withColumn("driverId", F.col("driverId").cast("int"))
    .withColumn("constructorId", F.col("constructorId").cast("int"))
    .withColumn("number", F.when(F.col("number") == "\\N", None).otherwise(F.col("number")).cast("int"))
    .withColumn("grid", F.col("grid").cast("int"))
    .withColumn("position", F.when(F.col("position") == "\\N", None).otherwise(F.col("position")).cast("int"))
    .withColumn("positionOrder", F.col("positionOrder").cast("int"))
    .withColumn("points", F.col("points").cast("double"))
    .withColumn("laps", F.col("laps").cast("int"))
    .withColumn("milliseconds", F.when(F.col("milliseconds") == "\\N", None).otherwise(F.col("milliseconds")).cast("bigint"))
)

df_pitstops = (
    spark.read.csv("/Volumes/gr5069/raw/f1_data/pit_stops.csv", header=True)
    .withColumn("raceId", F.col("raceId").cast("int"))
    .withColumn("driverId", F.col("driverId").cast("int"))
    .withColumn("stop", F.col("stop").cast("int"))
    .withColumn("lap", F.col("lap").cast("int"))
    .withColumn("milliseconds", F.col("milliseconds").cast("bigint"))
)

print("drivers")
df_drivers.printSchema()

print("races")
df_races.printSchema()

print("results")
df_results.printSchema()

print("pit stops")
df_pitstops.printSchema()

### Code Explanation

I used `spark.read.csv(..., header=True)` to load each dataset as a Spark DataFrame.

Then I used:
- `withColumn()` to replace columns with corrected data types,
- `cast()` to convert numeric fields from string to integer or double,
- `to_date()` to convert date strings into Spark date columns.

This preprocessing step is important because later questions require mathematical operations, date comparisons, ranking, and aggregation, all of which depend on correct data types.

## Q1. Average pit stop time for each driver in each race

To answer this question, I use the `pit_stops` dataset.

My logic is:
1. group the pit stop records by `raceId` and `driverId`,
2. compute the average pit stop time for each driver in each race,
3. separately compute the fastest and slowest single pit stop in each race,
4. join these results together so each row shows:
   - the driver's average pit stop time in that race,
   - the fastest pit stop recorded in that race,
   - the slowest pit stop recorded in that race.

I use the `milliseconds` column because it is numeric and consistent for calculation.

In [0]:
# ---------- Q1: DRIVER AVERAGE PIT TIME PER RACE ----------
q1_driver_avg = (
    df_pitstops
    .groupBy("raceId", "driverId")
    .agg(
        F.avg("milliseconds").alias("avg_pit_ms")
    )
)

# ---------- Q1: FASTEST AND SLOWEST PIT STOP PER RACE ----------
q1_race_fast_slow = (
    df_pitstops
    .groupBy("raceId")
    .agg(
        F.min("milliseconds").alias("fastest_pit_ms"),
        F.max("milliseconds").alias("slowest_pit_ms")
    )
)

# ---------- ADD DRIVER / RACE NAMES ----------
q1_result = (
    q1_driver_avg
    .join(q1_race_fast_slow, on="raceId", how="left")
    .join(df_drivers.select("driverId", "forename", "surname"), on="driverId", how="left")
    .join(df_races.select("raceId", "year", "name", "date"), on="raceId", how="left")
    .select(
        "raceId", "year", "name", "date",
        "driverId", "forename", "surname",
        F.round("avg_pit_ms", 2).alias("avg_pit_ms"),
        "fastest_pit_ms",
        "slowest_pit_ms"
    )
    .orderBy("raceId", "avg_pit_ms")
)

display(q1_result)

### Code Explanation

I used `groupBy()` and `agg()` to summarize the pit stop data.

- `avg("milliseconds")` calculates the average pit stop duration for each driver in each race.
- `min("milliseconds")` finds the fastest pit stop in each race.
- `max("milliseconds")` finds the slowest pit stop in each race.

Then I used `join()` to attach:
- driver names from the `drivers` dataset,
- race information from the `races` dataset.

Finally, I used `round()` to make the average pit stop values easier to read and `orderBy()` to sort the output.

### EXTRA CREDIT

An alternative way to solve this question would be to use Spark SQL instead of the DataFrame API.  
For example, I could register each DataFrame as a temporary view and then write one SQL query with `GROUP BY`, `AVG`, `MIN`, and `MAX`.

Another alternative would be to compute the fastest and slowest pit stop per race using window functions instead of a separate grouped summary.

## Q2. Rank the average pit stop time using finishing order

I first reuse the average pit stop result from Question 1. Then I combine it with race results.

My logic is:
1. calculate each driver's average pit stop time in each race,
2. join this with the `results` dataset using `raceId` and `driverId`,
3. use finishing order to rank the drivers within each race.

For drivers who did not finish the race:
- I decided to keep them in the dataset if they have a `positionOrder`,
- but I note that finishing order for non-finishers is not the same as officially completing the race,
- so if the instructor prefers only completed races, I can filter using `statusId` or another completion indicator.

In this version, I keep all drivers with a valid `positionOrder` because it provides a consistent ranking field across the race.

In [0]:
# ---------- Q2: JOIN AVERAGE PIT TIME WITH RESULTS ----------
q2_base = (
    q1_driver_avg
    .join(
        df_results.select("raceId", "driverId", "position", "positionOrder", "statusId"),
        on=["raceId", "driverId"],
        how="left"
    )
    .join(df_drivers.select("driverId", "forename", "surname"), on="driverId", how="left")
    .join(df_races.select("raceId", "year", "name", "date"), on="raceId", how="left")
)

# Rank average pit stop time by finishing order within each race
q2_window = Window.partitionBy("raceId").orderBy(F.col("positionOrder").asc())

q2_result = (
    q2_base
    .withColumn("finish_rank_in_race", F.row_number().over(q2_window))
    .select(
        "raceId", "year", "name", "date",
        "driverId", "forename", "surname",
        F.round("avg_pit_ms", 2).alias("avg_pit_ms"),
        "position",
        "positionOrder",
        "statusId",
        "finish_rank_in_race"
    )
    .orderBy("raceId", "finish_rank_in_race")
)

display(q2_result)

### Code Explanation

I started from the average pit stop table created in Question 1 and joined it with selected columns from `results`.I found that some fields in `results.csv` use `\N` to represent missing values. Before converting those columns to integers, I replaced `\N` with null values. This avoids casting errors and makes the ranking step more robust.

The main commands used here are:
- `join()` to combine pit stop summaries with finishing information,
- `Window.partitionBy("raceId").orderBy("positionOrder")` to define race-level ranking,
- `row_number().over(window)` to create the finishing rank within each race.

I used `positionOrder` rather than `positionText` because `positionOrder` is numeric and easier to sort consistently.

Regarding drivers who did not finish, I kept them if they still had a `positionOrder` value. This keeps the ranking process reproducible and avoids manually excluding records unless the assignment specifically says to count only finishers.

### EXTRA CREDIT

An alternative way to answer this question is to keep only official finishers before ranking.  
This approach is stricter because it removes drivers who have a race record but did not officially finish.  
I identify finishers by requiring a non-null final position.

In [0]:
# Q2 EXTRA CREDIT: keep only official finishers
q2_finishers_only = (
    q1_driver_avg
    .join(
        df_results.select("raceId", "driverId", "position", "positionOrder", "statusId"),
        on=["raceId", "driverId"],
        how="inner"
    )
    .filter(F.col("position").isNotNull())
    .join(df_drivers.select("driverId", "forename", "surname"), on="driverId", how="left")
    .join(df_races.select("raceId", "year", "name", "date"), on="raceId", how="left")
)

q2_finishers_window = Window.partitionBy("raceId").orderBy(F.col("positionOrder").asc())

q2_finishers_result = (
    q2_finishers_only
    .withColumn("finish_rank_in_race", F.row_number().over(q2_finishers_window))
    .select(
        "raceId", "year", "name", "date",
        "driverId", "forename", "surname",
        F.round("avg_pit_ms", 2).alias("avg_pit_ms"),
        "position",
        "positionOrder",
        "statusId",
        "finish_rank_in_race"
    )
    .orderBy("raceId", "finish_rank_in_race")
)

display(q2_finishers_result)

### EXTRA CREDIT Explanation

In this version, I filtered out drivers whose final classified `position` is null.  
Then I ranked the remaining drivers within each race using `positionOrder`.

This produces a stricter interpretation of finishing order because it focuses only on officially classified finishers.

## Q3. Insert the missing three-letter driver code

The goal of this question is to fill in missing three-letter driver codes in the `drivers` dataset.

My logic is:
1. identify rows where the `code` column is null or blank,
2. create a replacement code from the driver's name,
3. use that generated value only when the original code is missing.

To generate a code, I use:
- the first three letters of the surname if possible,
- converted to uppercase.

This is a practical rule because official F1 driver codes are usually short uppercase abbreviations and are often strongly tied to the surname. I chose this method because it is systematic and reproducible. I also keep the original code when it already exists.

In [0]:
# ---------- Q3: FILL MISSING DRIVER CODES ----------
q3_result = (
    df_drivers
    .withColumn(
        "generated_code",
        F.upper(F.substring(F.col("surname"), 1, 3))
    )
    .withColumn(
        "final_code",
        F.when(
            F.col("code").isNull() | (F.trim(F.col("code")) == ""),
            F.col("generated_code")
        ).otherwise(F.col("code"))
    )
    .select(
        "driverId", "forename", "surname", "code", "generated_code", "final_code"
    )
    .orderBy("driverId")
)

display(q3_result)

### Code Explanation

I used:
- `substring()` to take the first three characters of the surname,
- `upper()` to convert them to uppercase,
- `when(...).otherwise(...)` to fill only the missing `code` values,
- `trim()` to handle blank strings as well as null values.

This approach preserves the existing official codes and only inserts values where the code is missing.

I arrived at these replacement codes by following a consistent naming rule rather than assigning values manually one by one. That makes the method transparent and easy to reproduce.

### EXTRA CREDIT

Another possible rule for missing driver codes is to use:
- the first letter of the forename, and
- the first two letters of the surname.

This creates a three-letter code that still follows a clear and reproducible naming rule.

In [0]:
# Q3 EXTRA CREDIT: alternative generated code rule
q3_alt_result = (
    df_drivers
    .withColumn(
        "alt_generated_code",
        F.upper(
            F.concat(
                F.substring(F.col("forename"), 1, 1),
                F.substring(F.col("surname"), 1, 2)
            )
        )
    )
    .withColumn(
        "alt_final_code",
        F.when(
            F.col("code").isNull() | (F.trim(F.col("code")) == ""),
            F.col("alt_generated_code")
        ).otherwise(F.col("code"))
    )
    .select(
        "driverId", "forename", "surname", "code",
        "alt_generated_code", "alt_final_code"
    )
    .orderBy("driverId")
)

display(q3_alt_result)

## Q4. Youngest and oldest driver for each race

For this question, I need each driver's age on the date of each race, not their age today.

My logic is:
1. join `results`, `drivers`, and `races` so that each race-driver record contains both the driver's date of birth and the race date,
2. create an `Age` column that counts how many birthdays the driver had already had by that race date,
3. rank drivers by age within each race,
4. select the youngest and oldest driver in each race.

I define age as the number of full years lived as of the race date. This is more accurate than simply dividing day differences by 365.

### EXTRA CREDIT Explanation

I used `concat()` to combine:
- one character from the forename,
- and two characters from the surname.

Then I converted the result to uppercase and used it only when the original code was missing.

This method is an alternative to using the first three letters of the surname.

In [0]:
# ---------- Q4: BUILD RACE-DRIVER AGE TABLE ----------
q4_base = (
    df_results
    .select("raceId", "driverId")
    .dropDuplicates()
    .join(df_drivers.select("driverId", "forename", "surname", "dob"), on="driverId", how="left")
    .join(df_races.select("raceId", "year", "name", "date"), on="raceId", how="left")
)

# Accurate full-year age calculation as of race date
q4_base = (
    q4_base
    .withColumn("race_year", F.year("date"))
    .withColumn("birth_year", F.year("dob"))
    .withColumn(
        "had_birthday",
        F.when(
            (F.month("date") > F.month("dob")) |
            ((F.month("date") == F.month("dob")) & (F.dayofmonth("date") >= F.dayofmonth("dob"))),
            1
        ).otherwise(0)
    )
    .withColumn(
        "Age",
        F.col("race_year") - F.col("birth_year") - (1 - F.col("had_birthday"))
    )
)

# youngest
youngest_window = Window.partitionBy("raceId").orderBy(F.col("Age").asc(), F.col("dob").desc())
q4_youngest = (
    q4_base
    .withColumn("rn", F.row_number().over(youngest_window))
    .filter(F.col("rn") == 1)
    .select(
        "raceId", "year", "name", "date",
        F.col("driverId").alias("youngest_driverId"),
        F.col("forename").alias("youngest_forename"),
        F.col("surname").alias("youngest_surname"),
        F.col("Age").alias("youngest_age")
    )
)

# oldest
oldest_window = Window.partitionBy("raceId").orderBy(F.col("Age").desc(), F.col("dob").asc())
q4_oldest = (
    q4_base
    .withColumn("rn", F.row_number().over(oldest_window))
    .filter(F.col("rn") == 1)
    .select(
        "raceId",
        F.col("driverId").alias("oldest_driverId"),
        F.col("forename").alias("oldest_forename"),
        F.col("surname").alias("oldest_surname"),
        F.col("Age").alias("oldest_age")
    )
)

q4_result = (
    q4_youngest
    .join(q4_oldest, on="raceId", how="inner")
    .orderBy("raceId")
)

display(q4_result)

### Code Explanation

I joined:
- `results` to get which drivers appeared in each race,
- `drivers` to get each driver's date of birth,
- `races` to get the race date.

Then I calculated age in full years using:
- `year()` to compare birth year and race year,
- `month()` and `dayofmonth()` to check whether the birthday had already happened by the race date,
- `when()` to create a flag called `had_birthday`.

This is more accurate than using `datediff()/365`, because that method only approximates age and does not correctly count birthdays.

Finally, I used two window functions:
- one ordered by age ascending to get the youngest driver,
- one ordered by age descending to get the oldest driver.

### EXTRA CREDIT

Another way to calculate age is to use the number of months between the race date and the date of birth, divide by 12, and then take the floor.

This is shorter than the birthday-flag method, although the birthday-based definition is easier to explain in words.

In [0]:
# Q4 EXTRA CREDIT: age via months_between
q4_alt_base = (
    df_results
    .select("raceId", "driverId")
    .dropDuplicates()
    .join(df_drivers.select("driverId", "forename", "surname", "dob"), on="driverId", how="left")
    .join(df_races.select("raceId", "year", "name", "date"), on="raceId", how="left")
    .withColumn("Age_alt", F.floor(F.months_between(F.col("date"), F.col("dob")) / 12))
)

q4_alt_youngest_window = Window.partitionBy("raceId").orderBy(F.col("Age_alt").asc(), F.col("dob").desc())
q4_alt_oldest_window = Window.partitionBy("raceId").orderBy(F.col("Age_alt").desc(), F.col("dob").asc())

q4_alt_youngest = (
    q4_alt_base
    .withColumn("rn", F.row_number().over(q4_alt_youngest_window))
    .filter(F.col("rn") == 1)
    .select(
        "raceId", "year", "name", "date",
        F.col("driverId").alias("youngest_driverId"),
        F.col("forename").alias("youngest_forename"),
        F.col("surname").alias("youngest_surname"),
        F.col("Age_alt").alias("youngest_age_alt")
    )
)

q4_alt_oldest = (
    q4_alt_base
    .withColumn("rn", F.row_number().over(q4_alt_oldest_window))
    .filter(F.col("rn") == 1)
    .select(
        "raceId",
        F.col("driverId").alias("oldest_driverId"),
        F.col("forename").alias("oldest_forename"),
        F.col("surname").alias("oldest_surname"),
        F.col("Age_alt").alias("oldest_age_alt")
    )
)

q4_alt_result = (
    q4_alt_youngest
    .join(q4_alt_oldest, on="raceId", how="inner")
    .orderBy("raceId")
)

display(q4_alt_result)

### EXTRA CREDIT Explanation

Here I used `months_between(date, dob)` to measure the total number of months between birth and race date.  
Then I divided by 12 and applied `floor()` to get completed years.

This produces a compact age calculation and can be used as an alternative check against the birthday-based age method.

## Q5. Prior podium counts for each driver at any given race

The README asks for three new columns that show, for each race, how many times a driver had already finished:
- 1st,
- 2nd,
- 3rd

before that race.

My logic is:
1. take the race results and race dates,
2. create indicator columns for whether the driver finished 1st, 2nd, or 3rd in that race,
3. use a window ordered by race date for each driver,
4. compute cumulative sums up to the previous row only.

This gives each driver’s prior podium history at the time of each race.

In [0]:
# ---------- Q5: BUILD BASE TABLE ----------
q5_base = (
    df_results
    .join(df_races.select("raceId", "year", "round", "date", "name"), on="raceId", how="left")
    .join(df_drivers.select("driverId", "forename", "surname"), on="driverId", how="left")
    .select(
        "raceId", "driverId", "forename", "surname",
        "year", "round", "date", "name", "positionOrder"
    )
    .withColumn("is_win", F.when(F.col("positionOrder") == 1, 1).otherwise(0))
    .withColumn("is_second", F.when(F.col("positionOrder") == 2, 1).otherwise(0))
    .withColumn("is_third", F.when(F.col("positionOrder") == 3, 1).otherwise(0))
)

# Window: all previous races for each driver
q5_window = (
    Window
    .partitionBy("driverId")
    .orderBy("date", "raceId")
    .rowsBetween(Window.unboundedPreceding, -1)
)

q5_result = (
    q5_base
    .withColumn("prior_wins", F.coalesce(F.sum("is_win").over(q5_window), F.lit(0)))
    .withColumn("prior_second_places", F.coalesce(F.sum("is_second").over(q5_window), F.lit(0)))
    .withColumn("prior_third_places", F.coalesce(F.sum("is_third").over(q5_window), F.lit(0)))
    .select(
        "raceId", "year", "round", "date", "name",
        "driverId", "forename", "surname", "positionOrder",
        "prior_wins", "prior_second_places", "prior_third_places"
    )
    .orderBy("driverId", "date")
)

display(q5_result)

### Code Explanation

I created three binary indicator columns:
- `is_win` = 1 when `positionOrder` is 1,
- `is_second` = 1 when `positionOrder` is 2,
- `is_third` = 1 when `positionOrder` is 3.

Then I used a window function partitioned by `driverId` and ordered by `date`.

The key part is:
- `rowsBetween(Window.unboundedPreceding, -1)`

This means the cumulative sum includes all earlier races for that driver, but not the current race itself. That is exactly what the question asks for: how many prior podium results the driver had at that moment in time.

I used `coalesce(..., 0)` so that drivers with no prior podiums start from zero instead of null.

### EXTRA CREDIT

Another way to compute prior podium counts is to use a self-join.

In this approach, I match each driver’s current race to all of that driver’s earlier races, then count how many of those earlier results were wins, second places, or third places.

In [0]:
# Q5 EXTRA CREDIT: self-join approach

q5_self_base = (
    df_results
    .join(df_races.select("raceId", "date", "year", "round", "name"), on="raceId", how="left")
    .join(df_drivers.select("driverId", "forename", "surname"), on="driverId", how="left")
    .select(
        "raceId", "driverId", "forename", "surname",
        "date", "year", "round", "name", "positionOrder"
    )
)

current_races = q5_self_base.alias("cur")
past_races = q5_self_base.alias("past")

q5_self_joined = (
    current_races
    .join(
        past_races,
        on=(
            (F.col("cur.driverId") == F.col("past.driverId")) &
            (F.col("past.date") < F.col("cur.date"))
        ),
        how="left"
    )
)

q5_self_result = (
    q5_self_joined
    .groupBy(
        F.col("cur.raceId").alias("raceId"),
        F.col("cur.driverId").alias("driverId"),
        F.col("cur.forename").alias("forename"),
        F.col("cur.surname").alias("surname"),
        F.col("cur.date").alias("date"),
        F.col("cur.year").alias("year"),
        F.col("cur.round").alias("round"),
        F.col("cur.name").alias("name"),
        F.col("cur.positionOrder").alias("positionOrder")
    )
    .agg(
        F.sum(F.when(F.col("past.positionOrder") == 1, 1).otherwise(0)).alias("prior_wins_selfjoin"),
        F.sum(F.when(F.col("past.positionOrder") == 2, 1).otherwise(0)).alias("prior_second_places_selfjoin"),
        F.sum(F.when(F.col("past.positionOrder") == 3, 1).otherwise(0)).alias("prior_third_places_selfjoin")
    )
    .orderBy("driverId", "date")
)

display(q5_self_result)

### EXTRA CREDIT Explanation

I created two aliases of the same race-result table:
- `cur` for the current race,
- `past` for earlier races.

Then I joined them on:
- the same driver,
- and `past.date < cur.date`.

After that, I counted how many earlier race results had `positionOrder` equal to 1, 2, or 3.

This produces the same kind of prior podium counts as the window-function solution, but through a self-join instead.

## Q6. My Own Question

### Question:
Which driver had the best average finishing position across all races they entered?

I chose this question because it can be answered clearly using the `results` dataset, and it gives a useful summary of driver performance over time.

My logic is:
1. use the finishing order of each driver in each race,
2. group by driver,
3. calculate the average finishing position,
4. sort from lowest average position to highest, because a lower finishing position is better.

In [0]:
q6_result = (
    df_results
    .join(df_drivers.select("driverId", "forename", "surname"), on="driverId", how="left")
    .groupBy("driverId", "forename", "surname")
    .agg(
        F.count("*").alias("num_races"),
        F.avg("positionOrder").alias("avg_finish_position")
    )
    .filter(F.col("num_races") > 0)
    .select(
        "driverId", "forename", "surname", "num_races",
        F.round("avg_finish_position", 2).alias("avg_finish_position")
    )
    .orderBy("avg_finish_position", "surname")
)

display(q6_result)

### Code Explanation

I joined `results` with `drivers` so the output would include driver names.

Then I grouped by driver and used:
- `count("*")` to count how many races each driver entered,
- `avg("positionOrder")` to calculate the average finishing position.

Because finishing 1st is better than finishing 10th, a lower average finishing position indicates better overall race performance. Finally, I sorted the results in ascending order.

### EXTRA CREDIT

A different self-posed question is:

**Which race had the highest average pit stop time?**

This question uses the pit stop data to compare how long pit stops tended to be across races.

In [0]:
# Q6 EXTRA CREDIT: alternative own question
q6_alt_result = (
    df_pitstops
    .groupBy("raceId")
    .agg(
        F.count("*").alias("num_pit_stops"),
        F.avg("milliseconds").alias("avg_pit_ms")
    )
    .join(df_races.select("raceId", "year", "name", "date"), on="raceId", how="left")
    .select(
        "raceId", "year", "name", "date", "num_pit_stops",
        F.round("avg_pit_ms", 2).alias("avg_pit_ms")
    )
    .orderBy(F.col("avg_pit_ms").desc())
)

display(q6_alt_result)

### EXTRA CREDIT Explanation

I grouped the pit stop table by race and used:
- `count("*")` to count the number of pit stop records,
- `avg("milliseconds")` to compute the average pit stop time.

Then I joined the result with the `races` table so that each row includes the race name and date.  
Finally, I sorted in descending order of average pit stop time so the race with the highest average pit stop duration appears first.

## Conclusion

In this notebook, I used PySpark to answer six F1 data questions involving:
- grouped aggregation,
- joins across multiple datasets,
- date-based age calculation,
- ranking with window functions,
- cumulative historical counts,
- and an additional exploratory performance question.

These tasks show how Spark can be used to process structured racing data efficiently and reproducibly.